In [1]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)

n = 300
cities = ['Mumbai', 'Pune', 'Delhi', 'Bengaluru']

localities = {
    'Mumbai': ['Andheri', 'Bandra', 'Powai'],
    'Pune': ['Kothrud', 'Baner'],
    'Delhi': ['Dwarka', 'Rohini'],
    'Bengaluru': ['Whitefield', 'Indiranagar']
}

city = rng.choice(cities, n)
loc = [rng.choice(localities[c]) for c in city]
area = rng.integers(400, 2500, n).astype(float)
price = (area * rng.uniform(6000, 15000, n)).round(-3)

df = pd.DataFrame({
    'PropertyID': [f'P{1000+i}' for i in range(n)],
    'City': city,
    'Locality': loc,
    'BHK': rng.choice([1, 2, 3, 4], n).astype(float),
    'Area_sqft': area,
    'Price': price,
    'ListingDate': pd.to_datetime('2024-01-01') +
                   pd.to_timedelta(rng.integers(0, 600, n), unit='D')
})

variants = {
    'Mumbai': ['  mumbai', 'MUMBAI', 'Bombay', 'mumbai '],
    'Pune': ['pune ', 'PUNE', 'Poona'],
    'Delhi': ['delhi', 'New Delhi', ' DELHI'],
    'Bengaluru': ['bangalore', 'Bangalore', 'BENGALURU ']
}

for i in rng.choice(n, 90, replace=False):
    df.loc[i, 'City'] = rng.choice(variants[df.loc[i, 'City']])

for i in rng.choice(n, 40, replace=False):
    df.loc[i, 'Locality'] = rng.choice([
        df.loc[i, 'Locality'].lower(),
        df.loc[i, 'Locality'].upper(),
        ' ' + df.loc[i, 'Locality']
    ])

fmts = ['%Y-%m-%d', '%d/%m/%Y', '%d-%b-%Y', '%m.%d.%Y']
df['ListingDate'] = [
    d.strftime(rng.choice(fmts)) for d in df['ListingDate']
]

for i in rng.choice(n, 8, replace=False):
    df.loc[i, 'ListingDate'] = rng.choice([
        'not available', '32/13/2024', 'TBD'
    ])

for col, k in [
    ('BHK', 25),
    ('Area_sqft', 20),
    ('Price', 15),
    ('Locality', 12),
    ('City', 6)
]:
    df.loc[rng.choice(n, k, replace=False), col] = np.nan

idx = rng.choice(n, 8, replace=False)
df.loc[idx[:4], 'Price'] = df['Price'].max() * rng.uniform(4, 8, 4)
df.loc[idx[4:], 'Price'] = rng.integers(1000, 50000, 4)

df = pd.concat([df, df.sample(25, random_state=1)], ignore_index=True)
df.to_csv('raw_real_estate.csv', index=False)

print("Dirty dataset generated successfully!")
print("Shape:", df.shape)
print(df.head())

Dirty dataset generated successfully!
Shape: (325, 7)
  PropertyID       City     Locality  BHK  Area_sqft       Price  \
0      P1000     Mumbai       Bandra  3.0     1694.0  10810000.0   
1      P1001  Bengaluru  indiranagar  2.0     2035.0  28403000.0   
2      P1002      Delhi       Dwarka  1.0     1951.0  24165000.0   
3      P1003       Pune        Baner  2.0      682.0   5155000.0   
4      P1004       Pune        Baner  3.0     2134.0  14566000.0   

     ListingDate  
0    04-Feb-2024  
1  not available  
2     29/09/2024  
3     04/06/2024  
4     03.17.2024  


In [2]:
raw = pd.read_csv('raw_real_estate.csv')

print("=== Dataset Information ===")
raw.info()

print("\n=== Missing Values ===")
print(raw.isnull().sum())

print("\n=== Statistical Summary ===")
print(raw.describe())

print("\n=== Duplicate Rows ===")
print(raw.duplicated().sum())

print("\n=== Unique City Values ===")
print(raw['City'].unique())

=== Dataset Information ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PropertyID   325 non-null    object 
 1   City         318 non-null    object 
 2   Locality     310 non-null    object 
 3   BHK          296 non-null    float64
 4   Area_sqft    305 non-null    float64
 5   Price        310 non-null    float64
 6   ListingDate  325 non-null    object 
dtypes: float64(3), object(4)
memory usage: 17.9+ KB

=== Missing Values ===
PropertyID      0
City            7
Locality       15
BHK            29
Area_sqft      20
Price          15
ListingDate     0
dtype: int64

=== Statistical Summary ===
              BHK    Area_sqft         Price
count  296.000000   305.000000  3.100000e+02
mean     2.523649  1444.668852  1.789899e+07
std      1.107501   620.011894  2.526124e+07
min      1.000000   410.000000  3.763200e+04
25%      2.000000   

In [3]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return lower, upper

price_lower, price_upper = iqr_bounds(raw['Price'].dropna())

price_per_sqft = raw['Price'] / raw['Area_sqft']
pps_lower, pps_upper = iqr_bounds(price_per_sqft.dropna())

print("Price IQR bounds:", price_lower, "to", price_upper)
print("Price per sqft IQR bounds:", pps_lower, "to", pps_upper)

outliers = (
    (raw['Price'] < price_lower) |
    (raw['Price'] > price_upper) |
    (price_per_sqft < pps_lower) |
    (price_per_sqft > pps_upper)
)

print("Potential outliers:", outliers.sum())

Price IQR bounds: -10361750.0 to 40366250.0
Price per sqft IQR bounds: 1144.420335893993 to 19765.717128419856
Potential outliers: 8


In [4]:
clean = raw.copy()

# Remove duplicates
clean = clean.drop_duplicates()
clean = clean.drop_duplicates(subset='PropertyID', keep='first')

# Standardize text
for col in ['City', 'Locality']:
    clean[col] = clean[col].str.strip().str.title()

# Standardize city names
clean['City'] = clean['City'].replace({
    'Bombay': 'Mumbai',
    'Poona': 'Pune',
    'New Delhi': 'Delhi',
    'Bangalore': 'Bengaluru'
})

# Parse mixed date formats
def parse_date(value):
    if pd.isna(value):
        return pd.NaT

    for fmt in ['%Y-%m-%d', '%d/%m/%Y', '%d-%b-%Y', '%m.%d.%Y']:
        try:
            return pd.to_datetime(value, format=fmt)
        except (ValueError, TypeError):
            continue

    return pd.NaT

clean['ListingDate'] = clean['ListingDate'].apply(parse_date)

# Remove invalid dates and missing essential values
clean = clean.dropna(subset=['ListingDate', 'City', 'Price'])

# Handle missing values
clean['Locality'] = clean['Locality'].fillna('Unknown')
clean['BHK'] = clean['BHK'].fillna(clean['BHK'].mode()[0])

clean['Area_sqft'] = clean['Area_sqft'].fillna(
    clean.groupby('City')['Area_sqft'].transform('median')
)

print("Missing values after basic cleaning:")
print(clean.isnull().sum())

print("\nDataset shape after basic cleaning:", clean.shape)

Missing values after basic cleaning:
PropertyID     0
City           0
Locality       0
BHK            0
Area_sqft      0
Price          0
ListingDate    0
dtype: int64

Dataset shape after basic cleaning: (272, 7)


In [5]:
clean['Price_per_sqft'] = clean['Price'] / clean['Area_sqft']

price_lower, price_upper = iqr_bounds(clean['Price'])
pps_lower, pps_upper = iqr_bounds(clean['Price_per_sqft'])

print("Price bounds:", price_lower, "to", price_upper)
print("Price per sqft bounds:", pps_lower, "to", pps_upper)

clean['Price_Outlier'] = (
    (clean['Price'] < price_lower) |
    (clean['Price'] > price_upper) |
    (clean['Price_per_sqft'] < pps_lower) |
    (clean['Price_per_sqft'] > pps_upper)
)

print("Outliers detected:", clean['Price_Outlier'].sum())

clean = clean[~clean['Price_Outlier']].copy()
clean = clean.drop(columns=['Price_Outlier'])

# Convert data types and create derived feature
clean['BHK'] = clean['BHK'].astype(int)
clean['Price_per_sqft'] = clean['Price_per_sqft'].round(0)

clean = clean.reset_index(drop=True)
clean.to_csv('cleaned_real_estate.csv', index=False)

print("\nCleaned dataset saved successfully!")
print(clean.head())
print("Final shape:", clean.shape)

Price bounds: -10985125.0 to 41459875.0
Price per sqft bounds: 986.427330639558 to 20132.745252380693
Outliers detected: 10

Cleaned dataset saved successfully!
  PropertyID       City    Locality  BHK  Area_sqft       Price ListingDate  \
0      P1000     Mumbai      Bandra    3     1694.0  10810000.0  2024-02-04   
1      P1002      Delhi      Dwarka    1     1951.0  24165000.0  2024-09-29   
2      P1003       Pune       Baner    2      682.0   5155000.0  2024-06-04   
3      P1004       Pune       Baner    3     2134.0  14566000.0  2024-03-17   
4      P1005  Bengaluru  Whitefield    1     1525.0  11669000.0  2024-01-28   

   Price_per_sqft  
0          6381.0  
1         12386.0  
2          7559.0  
3          6826.0  
4          7652.0  
Final shape: (262, 8)


In [6]:
raw = pd.read_csv('raw_real_estate.csv')
clean = pd.read_csv('cleaned_real_estate.csv')

comparison = pd.DataFrame({
    'Metric': [
        'Total Rows',
        'Missing Values',
        'Duplicate Rows',
        'Unique City Names',
        'Invalid Listing Dates'
    ],
    'Before Cleaning': [
        len(raw),
        raw.isnull().sum().sum(),
        raw.duplicated().sum(),
        raw['City'].nunique(),
        raw['ListingDate'].apply(parse_date).isnull().sum()
    ],
    'After Cleaning': [
        len(clean),
        clean.isnull().sum().sum(),
        clean.duplicated().sum(),
        clean['City'].nunique(),
        pd.to_datetime(clean['ListingDate'], errors='coerce').isnull().sum()
    ]
})

print("=== BEFORE AND AFTER CLEANING ===")
print(comparison.to_string(index=False))

comparison.to_csv('before_after_comparison.csv', index=False)

print("\n=== Price Statistics ===")
print(pd.concat([
    raw['Price'].describe().rename('Before Cleaning'),
    clean['Price'].describe().rename('After Cleaning')
], axis=1).round(2))

print("\nFinal City Names:", sorted(clean['City'].dropna().unique()))
print("\nAll output CSV files generated successfully.")

=== BEFORE AND AFTER CLEANING ===
               Metric  Before Cleaning  After Cleaning
           Total Rows              325             262
       Missing Values               86               0
       Duplicate Rows               25               0
    Unique City Names               17               4
Invalid Listing Dates                8               0

=== Price Statistics ===
       Before Cleaning  After Cleaning
count     3.100000e+02          262.00
mean      1.789899e+07     15512885.50
std       2.526124e+07      7818140.51
min       3.763200e+04      3075000.00
25%       8.661250e+06      9054750.00
50%       1.425450e+07     14447500.00
75%       2.134325e+07     21343250.00
max       2.706789e+08     35582000.00

Final City Names: ['Bengaluru', 'Delhi', 'Mumbai', 'Pune']

All output CSV files generated successfully.
